# Phase 2 — Leakage-safe splits

**Why (paper §3.4):** to know if the model *learned haze* rather than *memorised
places*, we control how photos are divided into **train / calibration / test**
(65% / 15% / 20%). We compare four strategies:

- **random** — the *leaky control*: near-duplicate photos from one place can land on
  both sides, inflating scores.
- **station_grouped** — all photos from a station go to one split → tested on unseen
  places. **Our primary protocol.**
- **geographic** — whole regions held out (a tougher test).
- **temporal** — train on earlier years, test on later ones.

The **calibration** split is kept separate because Phase 6's guarantee depends on it.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh Colab session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

In [ ]:
from src.config import load_config
from src import data, splits
import pandas as pd
cfg = load_config()

SOURCE = cfg["data"]["drive_path"]        # laptop test: "tests/fixture_ds"
ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
print("clean rows:", len(df))

## Compare all four strategies

The key column is **`stations_straddling_splits`**: how many stations have photos in
more than one split. For a leakage-safe split this must be **0**. `random` and
`temporal` will show some straddling — that's the leakage we want to *measure*, not
hide.

In [ ]:
rows = []
for strat in ["random", "station_grouped", "geographic", "temporal"]:
    s = splits.make_splits(
        df, strategy=strat, fractions=(cfg["split"]["train"], cfg["split"]["calibration"], cfg["split"]["test"]),
        seed=cfg["seed"], station_col=cfg["data"]["station_col"], time_col=cfg["data"]["time_col"],
        lon_col=cfg["data"]["lon_col"], lat_col=cfg["data"]["lat_col"])
    r = splits.split_report(s, station_col=cfg["data"]["station_col"])
    rows.append({"strategy": strat, **r["counts"],
                 "straddling_stations": r["stations_straddling_splits"]})
pd.DataFrame(rows).set_index("strategy")

## See the geographic hold-out on a map

Colour each photo by which split it landed in under the **geographic** strategy. Whole
regions are one colour — the test regions are places the model never trains on.

In [ ]:
import matplotlib.pyplot as plt
g = splits.make_splits(df, strategy="geographic", seed=cfg["seed"],
                       lon_col=cfg["data"]["lon_col"], lat_col=cfg["data"]["lat_col"])
colors = {"train": "#4C78A8", "cal": "#F58518", "test": "#E45756"}
plt.figure(figsize=(9,4))
for name, c in colors.items():
    sub = g[g.split == name]
    plt.scatter(sub[cfg["data"]["lon_col"]], sub[cfg["data"]["lat_col"]], s=6, alpha=0.5, c=c, label=name)
plt.legend(); plt.xlabel("longitude"); plt.ylabel("latitude")
plt.title("Geographic split — whole regions held out"); plt.show()

## What this sets up

We'll train and evaluate under **station_grouped** (primary) and report **random**
alongside — the gap between them is an honest measurement of how much near-duplicate
leakage inflates results.

**Next:** `03_physics_features.ipynb` — turning each photo into the 5-channel input
(RGB + transmission + inverted saturation).